# 模块5：比赛提交与结果优化教学
---
## 🎯 学习目标
学完本模块你将能够：
- 掌握比赛结果生成的全流程和格式要求
- 理解阈值选择对比赛结果的影响，能够根据场景选择最优阈值
- 掌握TopN策略的适用场景和调优方法
- 学会比赛提分的核心技巧，能够独立参加类似的推荐算法比赛
- 理解线上线下结果不一致的常见原因和解决方法

---
## 📚 知识点讲解：比赛提交基础

### 1. 比赛任务回顾
我们的任务是预测用户在未来7天内是否会购买商品子集的商品，最终需要提交预测为会购买的用户-商品对列表。

### 2. 提交格式要求
阿里天池比赛要求提交的csv文件格式如下：
- 包含两列：user_id, item_id
- 没有表头，不要包含列名
- 每个用户-商品对占一行，不能重复
- 文件大小不超过20MB

### 3. 结果生成核心问题
模型输出的是用户购买概率（0~1之间的小数），我们需要确定一个阈值：
- 概率 >= 阈值 → 预测为购买，加入提交列表
- 概率 < 阈值 → 预测为不购买，不加入提交列表

阈值的选择直接决定了线上得分，是比赛最后阶段最关键的调优点。

---
## 💻 代码逐行拆解：结果生成全流程


In [ ]:
# 1. 导入需要的库
import pandas as pd
import numpy as np
import pickle

In [ ]:
# 2. 加载训练好的模型和全量特征数据
# 加载模型
with open("lgb_repurchase_model.pkl", "rb") as f:
    model = pickle.load(f)

# 加载全量特征数据（所有用户-商品对的特征）
df = pd.read_csv("final_train_dataset_fixed.csv")
feature_cols = [col for col in df.columns if col not in ["user_id", "item_id", "label"]]

print(f"总用户-商品对数量: {len(df):,}")
print(f"特征数量: {len(feature_cols)}")

⚠️ **注意：**
这里的全量特征数据是所有有过交互的用户-商品对，而不是我们之前采样的训练集。我们需要对所有可能的用户-商品对都做预测。

In [ ]:
# 3. 预测所有用户-商品对的购买概率
X = df[feature_cols]
df["pred_prob"] = model.predict(X, num_iteration=model.best_iteration)

print("概率预测完成！")
print(f"预测概率范围: [{df['pred_prob'].min():.4f}, {df['pred_prob'].max():.4f}]")
print(f"平均预测概率: {df['pred_prob'].mean():.4f}")

### 阈值选择策略
对于不平衡数据集，不能使用默认的0.5作为阈值，因为：
- 正样本比例只有约1%，大部分样本的预测概率都会低于0.5
- 使用0.5作为阈值会导致提交数量太少，召回率极低

我们有两种常用的阈值选择策略：
1. **固定阈值法**：选择一个较低的阈值（如0.03~0.1），保留所有概率大于阈值的样本
2. **TopN法**：按预测概率从高到低排序，取前N个样本作为预测结果

In [ ]:
# 策略1：固定阈值法
threshold = 0.05  # 阈值可以在0.03~0.1之间调整
predictions = df[df["pred_prob"] >= threshold][["user_id", "item_id"]]

# 策略2：TopN法
# top_n = 80000  # 取前8万个预测概率最高的样本
# predictions = df.sort_values("pred_prob", ascending=False).head(top_n)[["user_id", "item_id"]]

# 去重（比赛要求不能有重复的用户-商品对）
predictions = predictions.drop_duplicates(subset=["user_id", "item_id"], keep="first")

print(f"预测为购买的用户-商品对数量: {len(predictions):,}")
print(f"预计文件大小: {len(predictions)*30/1024/1024:.2f} MB （远小于20M限制）")

**两种策略对比：**
| 策略 | 优点 | 缺点 | 适用场景 |
|------|------|------|----------|
| 固定阈值法 | 简单直观，符合概率的物理意义 | 提交数量随模型变化而变化 | 模型比较稳定，对提交数量没有严格限制 |
| TopN法 | 提交数量固定，方便控制 | 阈值随模型变化，没有物理意义 | 比赛对提交数量有限制，或者需要固定提交量 |

💡 调优技巧：可以同时尝试两种策略，提交到线上看哪种效果更好，一般TopN法在比赛中表现更稳定。

In [ ]:
# 4. 生成符合比赛要求的提交文件
# 注意：不要保存表头，不要保存索引
predictions.to_csv(
    "tianchi_mobile_recommendation_predict.csv",
    index=False,
    header=False,
    encoding="utf-8"
)

print("提交文件生成完成！文件名：tianchi_mobile_recommendation_predict.csv")
print("可以直接上传到天池比赛平台提交。")

---
## 🚀 比赛提分核心技巧

### 1. 阈值调优
阈值是影响线上得分最关键的因素，建议按照以下步骤调优：
1. 首先在本地测试集上找到F1-score最高的阈值作为基准
2. 以这个基准为中心，上下浮动尝试多个阈值（比如基准是0.05，尝试0.03、0.04、0.05、0.06、0.07）
3. 分别提交这些阈值的结果，观察线上得分变化，找到最优阈值
4. 一般线上最优阈值会比本地最优阈值低一些，因为线上数据分布和本地可能有差异

### 2. 模型融合
模型融合是比赛提分的神器，简单的融合就能带来2~5个百分点的提升：
```python
# 简单加权融合示例
df['pred_prob'] = 0.7 * model1_pred + 0.3 * model2_pred
```
常用的融合方法：
- 简单平均/加权平均：适合同类型模型
- 投票法：适合多模型结果融合
- Stacking：多层模型融合，适合高手玩家

### 3. 后处理规则
结合业务规则对预测结果进行过滤，能够显著提升效果：
1. **过滤重复购买**：如果用户在特征期内已经购买过该商品，可以过滤掉（除非是高频复购品类）
2. **强特征过滤**：用户对该商品连浏览都没有过，预测为购买的可以过滤掉
3. **品类过滤**：有些品类几乎没有复购，可以过滤掉这些品类的预测结果

示例代码：
```python
# 过滤已经购买过的用户-商品对
bought_pairs = set(df_feat[df_feat['behavior_type'] == 4][['user_id', 'item_id']].apply(tuple, axis=1))
predictions['is_bought'] = predictions.apply(lambda x: (x['user_id'], x['item_id']) in bought_pairs, axis=1)
predictions = predictions[predictions['is_bought'] == False][['user_id', 'item_id']]
```

### 4. 线上线下不一致问题解决
经常会遇到线下分数很高，但线上提交分数很低的情况，常见原因和解决方法：

| 原因 | 解决方法 |
|------|----------|
| 数据泄露 | 严格检查时间窗口，确保特征不会使用标签期的数据 |
| 分布差异 | 线上线下数据分布不一致，用更接近线上的验证集做验证 |
| 阈值不合适 | 多尝试不同的阈值，尤其是更低的阈值 |
| 过拟合 | 减少特征数量，增加正则化，使用更简单的模型 |
| 特征处理不一致 | 确保训练和预测时的特征处理逻辑完全一致 |


---
## 🎯 实战例题

请你独立完成以下练习：
1. 尝试不同的阈值（0.03、0.04、0.05、0.06、0.07），计算不同阈值下的本地F1-score，找到本地最优阈值
2. 尝试TopN策略，分别取Top 5万、8万、10万、12万的结果，对比本地F1-score的差异
3. 实现一个后处理规则：过滤掉特征期内用户已经购买过的商品，看是否能提升本地F1-score
4. 假设你有两个模型的预测结果，尝试用加权融合的方式融合两个结果，看是否能提升效果


---
## ✅ 小测验

### 选择题
1. 对于1:100的不平衡二分类问题，最合适的分类阈值是？
   A. 0.1
   B. 0.3
   C. 0.5
   D. 0.7

2. 以下哪个不是比赛提交文件的要求？
   A. 包含user_id和item_id两列
   B. 必须有表头
   C. 不能有重复的用户-商品对
   D. 文件大小不超过20MB

3. 线上线下分数不一致最不可能的原因是？
   A. 数据泄露
   B. 阈值不合适
   C. 模型太简单
   D. 特征处理不一致

### 简答题
1. 固定阈值法和TopN法各有什么优缺点？分别适合什么场景？
2. 列举至少3种比赛提分的常用技巧。


---
## 📝 重点总结 & 毕业寄语

### 本模块核心知识点：
1. **结果生成流程**：全量预测 → 阈值选择 → 后处理 → 生成符合格式的提交文件
2. **阈值调优**：固定阈值法和TopN法的使用和调优
3. **提分技巧**：阈值调优、模型融合、后处理规则三大核心提分手段
4. **问题排查**：线上线下不一致的常见原因和解决方法

---
### 🎓 毕业寄语
恭喜你完成了整个阿里天池新人实战项目的学习！你已经掌握了：
✅ 数据探索与可视化分析
✅ 推荐场景特征工程全流程
✅ LightGBM模型训练与调优
✅ 比赛结果生成与优化

现在你已经具备了独立参加推荐算法比赛的能力，接下来可以：
1. 参加更多的天池比赛，在实战中提升能力
2. 尝试更复杂的模型，比如深度学习推荐模型（DIN、DeepFM等）
3. 把学到的方法应用到工作中的实际业务问题上

数据科学的道路没有终点，继续加油，未来可期！🚀